# TabDPT Regressor — DIMER E2E tutorial

[GitHub](https://github.com/kurtvalcorza/tabdpt-regressor-pipeline) · [Open in Colab](https://colab.research.google.com/github/kurtvalcorza/tabdpt-regressor-pipeline/blob/main/tutorials/tabdpt_regressor_colab.ipynb) · [Model](https://huggingface.co/Layer6/TabDPT) · [Upstream](https://github.com/layer6ai-labs/TabDPT-inference)

**Profile:** `E2E` · **DIMER Notebook Specification:** 1.0

**Capability and learning contract.** This notebook runs supervised tabular **regression** through the repository API. TabDPT is an in-context foundation model: `fit()` fits repository preprocessing and registers labelled support context; it does **not** gradient-train or fine-tune the pretrained weights. The upstream project supplies TabDPT/weights; this repository adds immutable provenance, SHA-256 weight verification, schema-safe preprocessing, deterministic controls, DIMER runtime/artifact integration, and regression evaluation.

By the end, you will verify the exact model, load a public sample or gated BYOD CSV, validate/split data, compare a training-mean baseline, evaluate MAE/RMSE/R², score new rows, export CSV/JSON outputs, export the DIMER serving artifact, and reload it from serialized files.

**Boundaries.** Regression only; no classification, gradient fine-tuning, or calibrated per-prediction uncertainty intervals. Sample metrics are sanity evidence, not benchmark/production evidence; upstream pretraining overlap with the public sample cannot be ruled out.

**Prerequisites.** Python 3.10+; GPU recommended, CPU supported but slower; internet for repository/model acquisition unless cached. FlashAttention is disabled for portable T4 execution. BYOD stays in the runtime; do not upload restricted data to an unauthorized environment. Run setup before imports.


In [ ]:
from pathlib import Path
import shutil
REPO_DIR=Path("/content/tabdpt-regressor-pipeline")
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
!git clone -q https://github.com/kurtvalcorza/tabdpt-regressor-pipeline.git /content/tabdpt-regressor-pipeline
%pip install -q -r /content/tabdpt-regressor-pipeline/tutorials/requirements-colab.txt
%pip install -q --no-deps /content/tabdpt-regressor-pipeline
!git -C /content/tabdpt-regressor-pipeline rev-parse HEAD


## 1. Runtime and immutable model provenance

The pinned dependency set is installed before model imports. The next cell prints the effective runtime, resolves the immutable Hugging Face revision, and verifies the model weight SHA-256. A matching digest establishes byte integrity against this repository's expected artifact, not model quality or producer authenticity.


In [ ]:
import csv, io, json, platform
import importlib.metadata as md
import numpy as np, pandas as pd, torch
from tabdpt_regressor_pipeline import (
 TABDPT_HF_REPO,TABDPT_HF_REVISION,TABDPT_UPSTREAM_CODE_COMMIT,
 TABDPT_WEIGHT_FILENAME,TABDPT_WEIGHT_SHA256,TabDPTRegressionPipeline,
 export_artifact_bundle,load_verified_artifact,resolve_tabdpt_weights)
print("Python",platform.python_version(),"torch",md.version("torch"),"tabdpt",md.version("tabdpt"),
      "pandas",md.version("pandas"),"sklearn",md.version("scikit-learn"))
print("Device",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU","use_flash=False")
weights=resolve_tabdpt_weights()
print({"repo":TABDPT_HF_REPO,"revision":TABDPT_HF_REVISION,"upstreamCodeCommit":TABDPT_UPSTREAM_CODE_COMMIT,
       "filename":TABDPT_WEIGHT_FILENAME,"sha256":TABDPT_WEIGHT_SHA256,"verifiedPath":str(weights)})


## 2. Sample/BYOD, validation, and leakage-aware split

Default data is scikit-learn's public diabetes regression sample. Set `USE_BYOD=True` to upload one CSV containing unique feature names plus finite numeric `target`; raw CSV headers are checked before pandas can rename duplicates. Random 80/20 splitting assumes rows are sufficiently independent; use preserved temporal/group/spatial boundaries for leakage-sensitive data. No model selection uses this holdout. `context_size` bounds support used per prediction; if support exceeds it, the notebook reports the upstream seeded subsampling.


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
USE_BYOD=False # @param {type:"boolean"}
def read_csv_checked(raw):
 text=raw.decode("utf-8-sig"); header=next(csv.reader(io.StringIO(text)),[])
 dup=sorted({x for x in header if header.count(x)>1})
 if dup: raise ValueError(f"Duplicate CSV columns: {dup}")
 return pd.read_csv(io.BytesIO(raw))
if USE_BYOD:
 from google.colab import files
 up=files.upload()
 if len(up)!=1: raise ValueError("Upload exactly one CSV.")
 name,raw=next(iter(up.items()))
 if not name.lower().endswith(".csv"): raise ValueError("BYOD must be CSV.")
 frame=read_csv_checked(raw); data_source=f"user CSV: {name}"
else:
 frame=load_diabetes(as_frame=True).frame; data_source="scikit-learn diabetes sample"
TARGET="target"; SEED=42; CONTEXT_SIZE=512; N_ENSEMBLES=2; BATCH_SIZE=512
if frame.columns.duplicated().any() or TARGET not in frame: raise ValueError("Unique columns and target are required.")
y=pd.to_numeric(frame[TARGET],errors="coerce")
if y.isna().any() or not np.isfinite(y.to_numpy(float)).all() or y.nunique()<2: raise ValueError("Target must be finite, numeric, and non-constant.")
train,test=train_test_split(frame,test_size=.2,random_state=SEED)
print(data_source,train.shape,test.shape,f"effective support <= {min(len(train),CONTEXT_SIZE)}")
if len(train)>CONTEXT_SIZE: print("Support exceeds context_size; seeded upstream context subsampling applies.")


## 3. Baseline, in-context conditioning, and evaluation

The training-mean constant predictor is the trivial baseline. MAE is average absolute error in target units; RMSE is also in target units and weights large errors more; R² is relative to a constant-mean reference and can be negative. Values below are single-holdout tutorial estimates with no dispersion claim. `fit()` performs preprocessing fitting plus in-context conditioning, **not gradient training**. Seeds control supported sampling/RNG paths; bitwise determinism across hardware kernels is not promised.


In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
yt=test[TARGET].to_numpy(float); bp=np.full(len(test),train[TARGET].mean())
baseline={"mae":float(mean_absolute_error(yt,bp)),"rmse":float(np.sqrt(mean_squared_error(yt,bp))),"r2":float(r2_score(yt,bp))}
pipe=TabDPTRegressionPipeline(model_weight_path=weights,compile_model=False,use_flash=False,seed=SEED)
pipe.fit(train,target_column=TARGET,seed=SEED)
kw={"n_ensembles":N_ENSEMBLES,"context_size":CONTEXT_SIZE,"batch_size":BATCH_SIZE,"seed":SEED}
metrics=pipe.evaluate(test,**kw)
print("tutorial TabDPT",metrics); print("training-mean baseline",baseline)


## 4. New-data inference and machine-readable outputs

The target is removed before scoring. Repository-fitted preprocessing is reused and not refit. Predictions are point estimates only. `row_id` is kept outside the model feature schema and exported with predictions so outputs map back to inputs. Provenance records immutable model identity, runtime/configuration, data source, and split.


In [ ]:
new_rows=test.drop(columns=[TARGET]).head(8).copy()
pred=pipe.predict(new_rows,**kw)
out=pd.DataFrame({"row_id":new_rows.index.to_numpy(),"prediction":pred.to_numpy()})
OUT=Path("/content/tabdpt-tutorial-output"); OUT.mkdir(parents=True,exist_ok=True)
out.to_csv(OUT/"tabdpt_regression_predictions.csv",index=False)
(OUT/"tabdpt_regression_metrics.json").write_text(json.dumps({"tabdpt":metrics,"training_mean_baseline":baseline},indent=2)+"\n")
prov={"model":{"repo":TABDPT_HF_REPO,"revision":TABDPT_HF_REVISION,"filename":TABDPT_WEIGHT_FILENAME,
"sha256":TABDPT_WEIGHT_SHA256,"upstreamCodeCommit":TABDPT_UPSTREAM_CODE_COMMIT},
"runtime":{"python":platform.python_version(),"torch":md.version("torch"),"tabdpt":md.version("tabdpt"),
"device":torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU","use_flash":False},
"data":{"source":data_source,"split":"single seeded 80/20 random holdout","seed":SEED},"inference":kw}
(OUT/"tabdpt_regression_provenance.json").write_text(json.dumps(prov,indent=2)+"\n")
print(out.head()); print("Wrote CSV + metrics/provenance JSON.")


## 5. Export and fresh-boundary verification

For this in-context model the deployable serving state is not the checkpoint alone: it includes the labelled support table and fitted preprocessing plus the exact pinned base-model contract. `training_context.parquet` inherits its source data's confidentiality/license/retention obligations. The artifact is copied to a fresh directory, validated, reconstructed from serialized state, and predictions are compared with explicit `rtol=1e-5`, `atol=1e-6`.


In [ ]:
ART=OUT/"artifact"; manifest_path=export_artifact_bundle(pipe,train,ART)
RELOAD=Path("/content/tabdpt-artifact-reload")
if RELOAD.exists(): shutil.rmtree(RELOAD)
shutil.copytree(ART,RELOAD)
reloaded=load_verified_artifact(RELOAD/"artifact.json",model_weight_path=weights,compile_model=False,use_flash=False,seed=SEED)
pred2=reloaded.predict(new_rows,**kw)
np.testing.assert_allclose(pred.to_numpy(),pred2.to_numpy(),rtol=1e-5,atol=1e-6)
print("PASS: serialized artifact reconstructed equivalent predictions (rtol=1e-5, atol=1e-6).")


## Interpretation, limits, and next steps

A successful run proves this repository can resolve/digest-check its pinned TabDPT weight, validate and condition on regression support data, compute sample metrics, score new rows, export machine-readable results and DIMER serving state, and reconstruct equivalent predictions from serialized files. It does **not** establish benchmark superiority, domain generalization, fairness, robustness, calibration, production safety, or deployment fitness. For real data, preserve domain-appropriate splits, compare task-relevant baselines, test representative inputs, and complete clean-runtime/on-platform release verification.
